In [ ]:
using CSV
using Glob
using DataFrames
using Statistics

using Plots
using Plots.PlotMeasures
using StatsPlots, KernelDensity

include("../src/foldexity.jl")


# Folding entropy

In [ ]:
folding = "ubiq2"
folding_trajpdb = "../../mdfolding/pdbtraj_$folding"
rms = CSV.read("../../mdfolding/$folding.dat", DataFrame, header=["f", "d"], skipto=3)

b, e, o = 3500, 4200, 1

x = rms.d[b:o:e] 

print(length(x))

plot(b:o:e, x, seriestype = :scatter, 
size = (600, 175), markersize = 2, alpha = 0.8, margins = 5mm,
    label="RMSD", xlabel = "Time", ylabel="RMSD (Å)")


In [ ]:
cpptraj("../../mdfolding/$folding.pdb", 
        "../../mdfolding/traj100_$folding.xtc", b, e, o,
        folding_trajpdb)

In [ ]:
traj3di = structure2fs3di(folding_trajpdb)
traj3di = traj3di[!, [:id, :seqaa, :seq3di]]
transform!(traj3di, :id => ByRow(s -> parse(Int, split(basename(s),".")[1])) => :id)
sort!(traj3di, :id)
size(traj3di)

In [ ]:
trajmu = structure2rsmu(folding_trajpdb)
transform!(trajmu, :id => ByRow(s -> parse(Int, split(basename(s),"_")[1])) => :id)
sort!(trajmu, :id)
size(trajmu)

In [ ]:
k=1
traj3di[!,:H1_fold ] = entropy_shannon.(traj3di.seq3di, k)
traj3di[!,:lz ] = lz76.(traj3di.seq3di, k)
trajmu[!,:H1_fold ] = entropy_shannon.(trajmu.seq, k)
trajmu[!,:lz ] = lz76.(trajmu.seq, k)
size(trajmu)

In [ ]:
function pltdots(x, label, mk=2, alpha=.5, wsize=10)
    
    moving_average = [mean(x[i:i+wsize]) for i in 1:length(x)-wsize]

    p = plot(x, seriestype = :scatter, markersize = mk, alpha = alpha, label=label, ylabel=label)
    p = plot!(moving_average, linewidth = 2, alpha = 1, label="moving average")

    return p
end

p1 = pltdots(traj3di.H1_fold, "H 3Di")
p2 = pltdots(trajmu.H1_fold, "H Mu")
p3 = pltdots(traj3di.lz, "LZ 3di" )
p4 = pltdots(trajmu.lz, "LZ Mu")

plot(p1, p2, p3, p4, layout = (4,1), size = (600, 600))

In [ ]:
fxdir(folding_trajpdb, "fxdata.tsv", 9, "seq", 4)
fx = CSV.read("fxdata.tsv", DataFrame, delim = "\t") ; rm("fxdata.tsv")
transform!(fx, :pdbpath => ByRow(s -> parse(Int, split(basename(s),".")[1])) => :id)
sort!(fx, :id)
fxseq = fx

size(fx)

pltdots(fxseq.fxity, "Fxity seq")


In [ ]:
fxdir(folding_trajpdb, "fxdata.tsv", 9, "knn", 20)
fx = CSV.read("fxdata.tsv", DataFrame, delim = "\t") ; rm("fxdata.tsv")
transform!(fx, :pdbpath => ByRow(s -> parse(Int, split(basename(s),".")[1])) => :id)
sort!(fx, :id)
fxknn = fx

size(fx)

pltdots(fxknn.fxity, "Fxity knn")

In [ ]:
p5 = pltdots(fxseq.fxity, "Fxity seq")
p6 = pltdots(fxknn.fxity, "Fxity knn")
p0 = pltdots(x, "RMSD")

plot(p5, p6, p0, layout=(3,1), size = (600, 500))

In [ ]:
ENV["PYTHON"] = "/vf/users/saakyanh2/micromamba/bin/python"
using Pkg; Pkg.build("PyCall")

In [ ]:
using PyCall

bdm = pyimport("pybdm")
bdm = bdm.BDM(ndim=1, nsymbols=4)


In [ ]:
# Step 1: Define amino acid to integer map
amino_acid_to_binary = Dict(
    'A' => "00001", 'C' => "00010", 'D' => "00011", 'E' => "00100",
    'F' => "00101", 'G' => "00110", 'H' => "00111", 'I' => "01000",
    'K' => "01001", 'L' => "01010", 'M' => "01011", 'N' => "01100",
    'P' => "01101", 'Q' => "01110", 'R' => "01111", 'S' => "10000",
    'T' => "10001", 'V' => "10010", 'W' => "10011", 'Y' => "10100"
)

# Function to convert a sequence to a binary string
function encode_to_binary(seq::String)
    join([amino_acid_to_binary[aa] for aa in seq])
end

function encode_to_bitarray(seq::String)
    reduce(vcat, [[parse(Int8, c) for c in amino_acid_to_binary[aa]] for aa in seq])
end


# Step 2: Function to convert sequence string to array of integers
function encode_sequence(seq::String)
    [amino_acid_to_int[aa] for aa in seq]
end


In [ ]:
traj3di.seqaa_int = [encode_to_bitarray(seq) for seq in df.seqaa]
traj3di.seq3di_int = [encode_to_bitarray(seq) for seq in df.seq3di]


In [ ]:
k=1
traj3di[!,:bdm3di] = mybdm.bdm.(traj3di.seq3di_int)
traj3di[!,:bdmaa] = mybdm.bdm.(traj3di.seqaa_int)
traj3di[!,:H_seqaa] = entropy_shannon.(traj3di.seqaa, k)
traj3di[!,:H_seq3di] = entropy_shannon.(traj3di.seq3di, k)

In [ ]:
function pltdots(x, label, mk=2, alpha=.5, wsize=10)
    
    moving_average = [mean(x[i:i+wsize]) for i in 1:length(x)-wsize]

    p = plot(x, seriestype = :scatter, markersize = mk, alpha = alpha, label=label, ylabel=label)
    #p = plot!(moving_average, linewidth = 2, alpha = 1, label="moving average")

    return p
end

p1 = pltdots(traj3di.H_seqaa, "H_seqaa")
p2 = pltdots(traj3di.H_seq3di, "H_seq3di")
p3 = pltdots(traj3di.bdmaa, "bdmaa" )
p4 = pltdots(traj3di.bdm3di, "bdm3di")

plot(p1, p2, p3, p4, layout = (4,1), size = (600, 600))